In [24]:
"""
Fractional Offset Quantization (FOQ) vs Hard Quantization
실제 OPT-125m pretrained weight로 WikiText-2 PPL 비교

설치:
    pip install torch transformers datasets

실행:
    python foq_experiment.py
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math, copy, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# ─────────────────────────────────────────
# 1. Quantizer
# ─────────────────────────────────────────

def hard_quantize(w: torch.Tensor, bits: int):
    """
    Hard (nearest-bin) quantization.
    저장: bin index k (bits bit)
    복원: b_k = w_min + k * delta
    """
    n  = 2 ** bits
    mn = w.min()
    mx = w.max()
    d  = ((mx - mn) / (n - 1)).clamp(min=1e-8)
    k  = torch.clamp(torch.floor((w - mn) / d), 0, n - 2).long()
    return mn + k.float() * d


def foq_b_quantize(w: torch.Tensor, bits: int, t_bits: int):
    """
    FOQ-B: bin index + fractional offset t 저장, 선형보간 복원.
    저장: k (bits bit) + t_q (t_bits bit) = (bits + t_bits) bit
    복원: w^ = (1-t_q)*b_k + t_q*b_{k+1}

    t = (w - b_k) / delta  ∈ [0, 1)
    t_q = round(t * 2^t_bits) / 2^t_bits  (t_bits 비트로 양자화)
    """
    n  = 2 ** bits
    mn = w.min()
    mx = w.max()
    d  = ((mx - mn) / (n - 1)).clamp(min=1e-8)
    k  = torch.clamp(torch.floor((w - mn) / d), 0, n - 2).long()
    bk = mn + k.float() * d          # 왼쪽 bin
    t  = ((w - bk) / d).clamp(0, 1)  # fractional offset ∈ [0,1)
    tq = (torch.round(t * (2 ** t_bits)) / (2 ** t_bits)).clamp(0, 1)
    return (1 - tq) * bk + tq * (bk + d)


def apply_ptq(model, bits: int, method: str, t_bits: int = 4):
    """
    모든 Linear layer의 weight에 PTQ 적용.
    bias는 건드리지 않음.
    """
    layer_mse = []
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            with torch.no_grad():
                w = module.weight.data.float()
                if method == 'hard':
                    wq = hard_quantize(w, bits)
                elif method == 'foq_b':
                    wq = foq_b_quantize(w, bits, t_bits)
                else:
                    raise ValueError(f"Unknown method: {method}")
                layer_mse.append((w - wq).pow(2).mean().item())
                module.weight.data = wq
    avg_mse = sum(layer_mse) / len(layer_mse)
    return model, avg_mse


# ─────────────────────────────────────────
# 2. Perplexity (WikiText-2)
# ─────────────────────────────────────────

@torch.no_grad()
def compute_perplexity(model, tokenizer, texts, max_len=512, device='cpu'):
    model.eval().to(device)
    total_nll = 0.0
    total_tok = 0

    for text in texts:
        enc = tokenizer(text, return_tensors='pt',
                        truncation=True, max_length=max_len)
        input_ids = enc['input_ids'].to(device)
        if input_ids.shape[1] < 2:
            continue
        out  = model(input_ids=input_ids, labels=input_ids)
        n    = input_ids.shape[1] - 1
        total_nll += out.loss.item() * n
        total_tok += n

    return math.exp(total_nll / total_tok)


# ─────────────────────────────────────────
# 3. 실험
# ─────────────────────────────────────────

def main():
    MODEL  = "facebook/opt-125m"
    BITS   = 4
    T_BITS = [1, 2, 4]       # FOQ-B의 t 저장 비트 수
    N_TEXT = 128               # 평가 텍스트 수 (많을수록 정확, 느림)
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

    print("=" * 64)
    print("  FOQ vs Hard Quantization — OPT-125m / WikiText-2")
    print(f"  device={DEVICE}, bits={BITS}, n_text={N_TEXT}")
    print("=" * 64)

    # ── 토크나이저 & 데이터 ──
    print("\n[데이터 로딩]")
    tokenizer = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
    dataset   = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    texts     = [t for t in dataset["text"] if len(t.strip()) > 100][:N_TEXT]
    print(f"  평가 샘플: {len(texts)}개")

    results = {}

    # ── FP32 Baseline ──
    print("\n[FP32 Baseline]")
    model_fp32 = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
    t0  = time.time()
    ppl = compute_perplexity(model_fp32, tokenizer, texts, device=DEVICE)
    print(f"  PPL = {ppl:.3f}  ({time.time()-t0:.1f}s)")
    results['FP32'] = {'ppl': ppl, 'total_bits': 32, 'mse': 0.0}
    del model_fp32

    # ── Hard Quantization ──
    print(f"\n[Hard Quant  {BITS}bit]")
    model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
    model, mse = apply_ptq(model, bits=BITS, method='hard')
    t0  = time.time()
    ppl = compute_perplexity(model, tokenizer, texts, device=DEVICE)
    print(f"  Avg weight MSE = {mse:.6f}")
    print(f"  PPL = {ppl:.3f}  ({time.time()-t0:.1f}s)")
    results['Hard'] = {'ppl': ppl, 'total_bits': BITS, 'mse': mse}
    del model

    # ── FOQ-B ──
    for tb in T_BITS:
        total_bits = BITS + tb
        label = f'FOQ-B +{tb}bit'
        print(f"\n[{label}  ({BITS}+{tb}={total_bits}bit/weight)]")
        model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
        model, mse = apply_ptq(model, bits=BITS, method='foq_b', t_bits=tb)
        t0  = time.time()
        ppl = compute_perplexity(model, tokenizer, texts, device=DEVICE)
        print(f"  Avg weight MSE = {mse:.6f}")
        print(f"  PPL = {ppl:.3f}  ({time.time()-t0:.1f}s)")
        results[label] = {'ppl': ppl, 'total_bits': total_bits, 'mse': mse}
        del model
    
    # foq_experiment.py 하단 main()에 추가
    for ref_bits in [6, 8]:
        print(f"\n[Hard Quant  {ref_bits}bit  (공정 비교용)]")
        model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
        model, mse = apply_ptq(model, bits=ref_bits, method='hard')
        ppl = compute_perplexity(model, tokenizer, texts, device=DEVICE)
        print(f"  MSE={mse:.6f}, PPL={ppl:.3f}")

    # ── 결과 테이블 ──
    print(f"\n{'─'*64}")
    print(f"{'방법':<22} {'bit/w':>6} {'Weight MSE':>12} {'PPL':>9} {'ΔPPL vs Hard':>14}")
    print(f"{'─'*64}")
    hard_ppl = results['Hard']['ppl']
    for name, r in results.items():
        d = f"{r['ppl']-hard_ppl:+.3f}" if name != 'Hard' else "(기준)"
        print(f"{name:<22} {r['total_bits']:>6} {r['mse']:>12.6f} {r['ppl']:>9.3f} {d:>14}")

    print(f"\n{'─'*64}")
    print("비트 효율 분석:")
    for tb in T_BITS:
        label = f'FOQ-B +{tb}bit'
        r = results[label]
        imp  = hard_ppl - r['ppl']
        ovhd = tb / BITS * 100
        print(f"  {label}: PPL {imp:+.3f}  (비트 오버헤드 +{ovhd:.0f}%,"
              f"  {BITS}bit→{BITS+tb}bit)")


if __name__ == "__main__":
    main()

  FOQ vs Hard Quantization — OPT-125m / WikiText-2
  device=cpu, bits=4, n_text=128

[데이터 로딩]
  평가 샘플: 128개

[FP32 Baseline]
  PPL = 52.333  (16.9s)

[Hard Quant  4bit]
  Avg weight MSE = 0.002339
  PPL = 9131.651  (19.0s)

[FOQ-B +1bit  (4+1=5bit/weight)]
  Avg weight MSE = 0.000143
  PPL = 7219.372  (18.9s)

[FOQ-B +2bit  (4+2=6bit/weight)]
  Avg weight MSE = 0.000036
  PPL = 141.755  (19.0s)

[FOQ-B +4bit  (4+4=8bit/weight)]
  Avg weight MSE = 0.000002
  PPL = 54.645  (19.3s)

[Hard Quant  6bit  (공정 비교용)]
  MSE=0.000131, PPL=39800.682

[Hard Quant  8bit  (공정 비교용)]
  MSE=0.000008, PPL=140.899

────────────────────────────────────────────────────────────────
방법                      bit/w   Weight MSE       PPL   ΔPPL vs Hard
────────────────────────────────────────────────────────────────
FP32                       32     0.000000    52.333      -9079.318
Hard                        4     0.002339  9131.651           (기준)
FOQ-B +1bit                 5     0.000143  7219.372      -1912

In [1]:
"""
FOQ-C vs STE — Quantization-Aware Training 비교
OPT-125m / WikiText-2

핵심 차이:
  STE:   backward에서 gradient를 그냥 통과 (∂/∂w = 1)
         codebook은 가장 가까운 bin b_k* 에만 gradient
  FOQ-C: backward에서 t 기반으로 인접 두 bin에 gradient 분배
         ∂/∂b_k*   = 1 - t
         ∂/∂b_k*+1 = t
         압축률은 동일 (k만 저장, 4bit)

설치:
    pip install torch transformers datasets

실행:
    python foq_c_experiment.py
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import math, copy, time
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

# ─────────────────────────────────────────
# 1. QAT Quantizer (학습 가능한 bin 위치)
# ─────────────────────────────────────────

class STEQuantizer(nn.Module):
    """
    기존 STE Quantizer.
    - forward: nearest bin (hard)
    - backward: gradient 그냥 통과 (identity)
    - bin 위치(scale, zero_point)는 학습 가능
    """
    def __init__(self, bits=4):
        super().__init__()
        self.bits = bits
        self.n    = 2 ** bits
        # 학습 가능한 scale, zero_point
        self.scale      = nn.Parameter(torch.tensor(1.0))
        self.zero_point = nn.Parameter(torch.tensor(0.0))

    def forward(self, w):
        # bin 위치 계산
        d  = self.scale.abs().clamp(min=1e-8)
        zp = self.zero_point
        # quantize
        w_scaled = (w - zp) / d
        k  = torch.clamp(torch.round(w_scaled), 0, self.n - 1)
        # dequantize (hard)
        w_q = k * d + zp
        # STE: backward는 w_q → w로 그냥 통과
        w_q = w + (w_q - w).detach()
        return w_q


class FOQCQuantizer(nn.Module):
    """
    FOQ-C Quantizer.
    - forward: nearest bin (hard, STE와 동일)
    - backward: t 기반으로 인접 두 bin에 gradient 분배
    - 저장: k만 (압축률 STE와 동일)
    - bin 위치(scale, zero_point)는 학습 가능
    """
    def __init__(self, bits=4):
        super().__init__()
        self.bits = bits
        self.n    = 2 ** bits
        self.scale      = nn.Parameter(torch.tensor(1.0))
        self.zero_point = nn.Parameter(torch.tensor(0.0))

    def forward(self, w):
        d  = self.scale.abs().clamp(min=1e-8)
        zp = self.zero_point

        w_scaled = (w - zp) / d
        k  = torch.clamp(torch.floor(w_scaled), 0, self.n - 2).detach()
        t  = (w_scaled - k).clamp(0, 1)  # fractional offset, gradient 흐름

        # 인접 두 bin
        b_k  = k * d + zp        # 왼쪽 bin
        b_k1 = (k + 1) * d + zp  # 오른쪽 bin

        # soft reconstruction (학습 시)
        # gradient: ∂/∂b_k = 1-t, ∂/∂b_k1 = t
        w_soft = (1 - t) * b_k + t * b_k1

        # forward는 hard와 동일하게 보이도록 detach 트릭
        # 실제 저장값: k (hard), gradient는 soft에서 흐름
        w_hard = b_k.detach() + (w - b_k).detach()
        # backward는 w_soft의 gradient 사용
        w_q = w_soft + (w_hard - w_soft).detach()
        return w_q


# ─────────────────────────────────────────
# 2. Quantized Linear Layer
# ─────────────────────────────────────────

class QuantizedLinear(nn.Module):
    """
    기존 Linear를 QAT용으로 감싸는 wrapper.
    weight에만 quantization 적용 (activation은 FP32 유지).
    """
    def __init__(self, linear: nn.Linear, quantizer: nn.Module):
        super().__init__()
        self.weight    = nn.Parameter(linear.weight.data.clone())
        self.bias      = nn.Parameter(linear.bias.data.clone()) if linear.bias is not None else None
        self.quantizer = quantizer

    def forward(self, x):
        w_q = self.quantizer(self.weight)
        return F.linear(x, w_q, self.bias)


def wrap_model_with_qat(model, quantizer_class, bits=4):
    """모든 Linear layer를 QuantizedLinear로 교체"""
    def _replace(module):
        for name, child in module.named_children():
            if isinstance(child, nn.Linear):
                q = quantizer_class(bits=bits)
                # scale 초기화: weight 범위 기반
                with torch.no_grad():
                    w = child.weight.data.float()
                    q.scale.data      = (w.max() - w.min()) / (2**bits - 1)
                    q.zero_point.data = w.min()
                setattr(module, name, QuantizedLinear(child, q))
            else:
                _replace(child)
    model_q = copy.deepcopy(model)
    _replace(model_q)
    return model_q


# ─────────────────────────────────────────
# 3. 학습 루프
# ─────────────────────────────────────────

def train_one_epoch(model, tokenizer, texts, optimizer,
                    max_len=128, device='cpu', max_steps=200):
    model.train().to(device)
    total_loss = 0.0
    steps = 0

    for text in texts:
        if steps >= max_steps:
            break
        enc = tokenizer(text, return_tensors='pt',
                        truncation=True, max_length=max_len)
        input_ids = enc['input_ids'].to(device)
        if input_ids.shape[1] < 2:
            continue

        optimizer.zero_grad()
        out  = model(input_ids=input_ids, labels=input_ids)
        loss = out.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        steps += 1

    return total_loss / max(steps, 1)


@torch.no_grad()
def compute_perplexity(model, tokenizer, texts, max_len=512, device='cpu'):
    model.eval().to(device)
    total_nll, total_tok = 0.0, 0
    for text in texts:
        enc = tokenizer(text, return_tensors='pt',
                        truncation=True, max_length=max_len)
        input_ids = enc['input_ids'].to(device)
        if input_ids.shape[1] < 2:
            continue
        out  = model(input_ids=input_ids, labels=input_ids)
        n    = input_ids.shape[1] - 1
        total_nll += out.loss.item() * n
        total_tok += n
    return math.exp(total_nll / total_tok)


# ─────────────────────────────────────────
# 4. 실험
# ─────────────────────────────────────────

def main():
    MODEL    = "facebook/opt-125m"
    BITS     = 4
    N_TRAIN  = 200   # 학습 텍스트 수
    N_EVAL   = 64    # 평가 텍스트 수
    EPOCHS   = 3
    LR       = 1e-4
    MAX_STEPS_PER_EPOCH = 100
    DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'

    print("=" * 64)
    print("  FOQ-C vs STE — QAT 비교 / OPT-125m / WikiText-2")
    print(f"  device={DEVICE}, bits={BITS}, epochs={EPOCHS}")
    print("=" * 64)

    # 데이터
    print("\n[데이터 로딩]")
    tokenizer  = AutoTokenizer.from_pretrained(MODEL, use_fast=True)
    dataset    = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
    eval_set   = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    train_texts = [t for t in dataset["text"]   if len(t.strip()) > 100][:N_TRAIN]
    eval_texts  = [t for t in eval_set["text"]  if len(t.strip()) > 100][:N_EVAL]
    print(f"  학습: {len(train_texts)}개 / 평가: {len(eval_texts)}개")

    # FP32 baseline
    print("\n[FP32 Baseline]")
    model_fp32 = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
    ppl_fp32   = compute_perplexity(model_fp32, tokenizer, eval_texts, device=DEVICE)
    print(f"  PPL = {ppl_fp32:.3f}")
    del model_fp32

    results = {'FP32': ppl_fp32}

    # STE & FOQ-C 비교
    for method_name, quantizer_class in [('STE', STEQuantizer), ('FOQ-C', FOQCQuantizer)]:
        print(f"\n{'='*40}")
        print(f"[{method_name}  {BITS}bit QAT]")

        # pretrained weight 로드 후 QAT wrap
        base  = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)
        model = wrap_model_with_qat(base, quantizer_class, bits=BITS)
        del base

        # quantizer parameter만 학습 (scale, zero_point)
        # 원본 weight도 같이 fine-tune
        optimizer = torch.optim.Adam(model.parameters(), lr=LR)

        # 초기 PPL (학습 전)
        ppl_init = compute_perplexity(model, tokenizer, eval_texts, device=DEVICE)
        print(f"  초기 PPL (학습 전) = {ppl_init:.3f}")

        ppl_history = [ppl_init]

        for epoch in range(1, EPOCHS + 1):
            t0   = time.time()
            loss = train_one_epoch(model, tokenizer, train_texts, optimizer,
                                   device=DEVICE, max_steps=MAX_STEPS_PER_EPOCH)
            ppl  = compute_perplexity(model, tokenizer, eval_texts, device=DEVICE)
            ppl_history.append(ppl)
            print(f"  epoch {epoch}/{EPOCHS}  loss={loss:.4f}  PPL={ppl:.3f}  ({time.time()-t0:.1f}s)")

        results[method_name] = ppl_history
        del model

    # 결과 요약
    print(f"\n{'='*64}")
    print("결과 요약")
    print(f"{'─'*64}")
    print(f"  FP32 baseline PPL: {results['FP32']:.3f}")
    print()
    print(f"  {'epoch':<8} {'STE PPL':>10} {'FOQ-C PPL':>12} {'ΔPPL(FOQ-STE)':>15}")
    print(f"  {'─'*48}")
    labels = ['초기'] + [f'epoch {i}' for i in range(1, EPOCHS+1)]
    for i, label in enumerate(labels):
        ste_ppl  = results['STE'][i]
        foqc_ppl = results['FOQ-C'][i]
        delta    = foqc_ppl - ste_ppl
        better   = "✓ FOQ-C" if delta < 0 else ""
        print(f"  {label:<8} {ste_ppl:>10.3f} {foqc_ppl:>12.3f} {delta:>+14.3f}  {better}")

    print(f"\n최종 결과:")
    print(f"  STE   최종 PPL: {results['STE'][-1]:.3f}")
    print(f"  FOQ-C 최종 PPL: {results['FOQ-C'][-1]:.3f}")
    improvement = results['STE'][-1] - results['FOQ-C'][-1]
    print(f"  FOQ-C 개선:     {improvement:+.3f}")
    if improvement > 0:
        print("  → FOQ-C가 STE 대비 PPL 개선 확인")
    else:
        print("  → 추가 분석 필요")


if __name__ == "__main__":
    main()


c:\Users\DMLab\anaconda3\envs\yunjun0914\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\DMLab\anaconda3\envs\yunjun0914\Lib\site-packages\transformers\utils\generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


  FOQ-C vs STE — QAT 비교 / OPT-125m / WikiText-2
  device=cpu, bits=4, epochs=3

[데이터 로딩]


c:\Users\DMLab\anaconda3\envs\yunjun0914\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  학습: 200개 / 평가: 64개

[FP32 Baseline]


c:\Users\DMLab\anaconda3\envs\yunjun0914\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
c:\Users\DMLab\anaconda3\envs\yunjun0914\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
W0513 11:47:18.740000 15000 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
c:\Users\DMLab\anaconda3\envs\yunjun0914\Lib\site-packages\transformers\utils\generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytre

  PPL = 58.983

[STE  4bit QAT]
  초기 PPL (학습 전) = 14632.591
  epoch 1/3  loss=7.5648  PPL=2090.924  (102.9s)
  epoch 2/3  loss=5.9269  PPL=1982.335  (102.4s)
  epoch 3/3  loss=5.1230  PPL=2427.921  (101.4s)

[FOQ-C  4bit QAT]


c:\Users\DMLab\anaconda3\envs\yunjun0914\Lib\site-packages\huggingface_hub\file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  초기 PPL (학습 전) = 58.983
  epoch 1/3  loss=4.0430  PPL=73.583  (216.2s)
  epoch 2/3  loss=2.4304  PPL=145.361  (215.2s)
  epoch 3/3  loss=1.5032  PPL=174.514  (217.2s)

결과 요약
────────────────────────────────────────────────────────────────
  FP32 baseline PPL: 58.983

  epoch       STE PPL    FOQ-C PPL   ΔPPL(FOQ-STE)
  ────────────────────────────────────────────────
  초기        14632.591       58.983     -14573.608  ✓ FOQ-C
  epoch 1    2090.924       73.583      -2017.341  ✓ FOQ-C
  epoch 2    1982.335      145.361      -1836.974  ✓ FOQ-C
  epoch 3    2427.921      174.514      -2253.407  ✓ FOQ-C

최종 결과:
  STE   최종 PPL: 2427.921
  FOQ-C 최종 PPL: 174.514
  FOQ-C 개선:     +2253.407
  → FOQ-C가 STE 대비 PPL 개선 확인
